 PakFreight Data Detective  Week 2: DW Lifecycle & Architectur
Name:shoukat hussain
Rollno:BSDSM-039
Section:7A



Course: Data Warehousing & Business Intelligence
Case:PakFreight moves cargo by Marine Rail and Road. Each mode has its own file its own format and its own idea of what a "shipment" is. Operations Finance and Sales all report a different total tonnage. This notebook finds out why builds one staging area, and proves the one true number 578.65 t

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import sqlite3
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# all four files sit next to the notebook
DATA = "."

 Part A The Data Detective Activity

 Loading the raw files (exactly as received)

Rule for staging: never edit the raw copy. I load everything as-is first, look at it, and only then build a separate cleaned copy.

Things I already know from opening the files:
- Marine: CSV, dates as `dd/mm/yyyy` text, weights in KG, status in UPPERCASE
- Rail: Excel with 2 title rows above the real header, weights in tons numeric IDs
- Road: CSV with lowercase ids, a duplicate, a blank row and a TOTAL row at the bottom

In [ ]:
# marine: read everything as text first so pandas doesn't guess the dates for me
marine_raw = pd.read_csv(f"{DATA}/marine_shipments.csv", dtype=str, keep_default_na=False)
marine_raw["Weight_KG"] = pd.to_numeric(marine_raw["Weight_KG"])
marine_raw["Cost_PKR"]  = pd.to_numeric(marine_raw["Cost_PKR"])
marine_raw

In [ ]:
# rail: header is on row 3 of the sheet, so skip the 2 title rows (same as Remove Top Rows in Power BI)
rail_raw = pd.read_excel(f"{DATA}/rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
rail_raw

In [ ]:
# what happens if I DON'T skip the title rows? (this is the Promoted Headers trap from the README)
bad = pd.read_excel(f"{DATA}/rail_shipments.xlsx", sheet_name="Rail", header=0)
print(list(bad.columns))
bad.head(4)

With `header=0` the title text becomes the column name and the real header turns into a data row. Every column would be named `Unnamed: n`. That's why the 2 rows must be removed before promoting headers.

In [ ]:
# road: keep as text again, the junk rows would otherwise get mixed types
road_raw = pd.read_csv(f"{DATA}/road_shipments.csv", dtype=str, keep_default_na=False)
road_raw.index = road_raw.index + 2   # +2 so the index matches the csv line number (line 1 = header)
road_raw

In [ ]:
# quick shape / dtype check on all three
for name, df in [("marine", marine_raw), ("rail", rail_raw), ("road", road_raw)]:
    print(f"--- {name}: {df.shape[0]} rows x {df.shape[1]} cols")
    print(df.dtypes.to_string())
    print()

## Task 1 — Data Detective: profile before you load

Before logging problems in a table I want to actually *prove* them with code, so I'm running a few profiling checks first.

In [ ]:
# ---- 1. hidden whitespace in the status columns
def has_space(s):
    return s != s.str.strip()

print("Marine status with leading/trailing spaces:", has_space(marine_raw["Status"]).sum())
print("Rail   status with leading/trailing spaces:", has_space(rail_raw["Status"]).sum())
print("Road   status with leading/trailing spaces:", has_space(road_raw["status"]).sum())

# ---- 2. how many different spellings of status do we get in each file?
print()
print("Marine:", sorted(marine_raw["Status"].unique()))
print("Rail  :", sorted(rail_raw["Status"].unique()))
print("Road  :", sorted(road_raw["status"].unique()))

In [ ]:
# ---- 3. same status after trim + case fix -> how many distinct values really exist?
all_status = pd.concat([marine_raw["Status"], rail_raw["Status"], road_raw["status"]])
print("raw distinct spellings :", all_status.nunique())
print("after strip + title    :", all_status.str.strip().str.title().replace("", np.nan).dropna().nunique())
print(sorted(all_status.str.strip().str.title().replace("", np.nan).dropna().unique()))

In [ ]:
# ---- 4. road: duplicates, blanks, total rows
dup_mask   = road_raw.duplicated(keep="first")
blank_mask = (road_raw.apply(lambda c: c.str.strip()) == "").all(axis=1)
total_mask = ~road_raw["trip_id"].str.match(r"(?i)^rd-\d+$") & ~blank_mask

print("exact duplicate rows  :", road_raw[dup_mask].index.tolist(),  road_raw[dup_mask]["trip_id"].tolist())
print("fully blank rows      :", road_raw[blank_mask].index.tolist())
print("non-shipment id rows  :", road_raw[total_mask].index.tolist(), road_raw[total_mask]["trip_id"].tolist())

In [ ]:
# ---- 5. units: compare typical weight per shipment in each file
print("Marine avg weight:", marine_raw["Weight_KG"].mean().round(0), "kg")
print("Road   avg weight:", pd.to_numeric(road_raw["weight_kg"], errors="coerce").iloc[:-2].mean().round(0), "kg")
print("Rail   avg weight:", rail_raw["Weight (Tons)"].mean().round(2), "<- looks tiny because it's TONS")

# ---- 6. dates: what does a US-locale parser do with the marine dates?
us_try = pd.to_datetime(marine_raw["Ship_Date"], format="%m/%d/%Y", errors="coerce")
uk_try = pd.to_datetime(marine_raw["Ship_Date"], format="%d/%m/%Y", errors="coerce")
chk = pd.DataFrame({"raw": marine_raw["Ship_Date"], "as_US(mm/dd)": us_try, "as_UK(dd/mm)": uk_try})
print()
print(chk.loc[[2, 11]])           # MR-1003 and MR-1012
print("rows that fail as US:", us_try.isna().sum())

In [ ]:
# ---- 7. id formats differ between sources -> same business key can't be matched
print("Marine ids:", marine_raw["Shipment_ID"].head(3).tolist())
print("Rail ids  :", rail_raw["Consignment No"].head(3).tolist(), "(numbers, no prefix)")
print("Road ids  :", road_raw["trip_id"].head(3).tolist(), "(lowercase)")

# ---- 8. column names - 3 different names for each business concept
pd.DataFrame({
    "Concept":     ["ID", "Date", "Origin", "Destination", "Weight", "Cost", "Status"],
    "Marine":      list(marine_raw.columns),
    "Rail":        list(rail_raw.columns),
    "Road":        list(road_raw.columns),
})

Now the log itself. I'm keeping the same columns as the T1 tab of the workbook: source file, where, problem, what goes wrong, which tier fixes it, how.

In [ ]:
issues = [
 # (source, where, problem, what goes wrong if loaded as-is, tier, fix)
 ("Marine", "Ship_Date (all rows)", "Dates are text in dd/mm/yyyy",
  "US-locale tool reads 03/02 as 2 March; 13/02/2026 becomes an error", "Staging", "Convert to a real date using the UK locale / explicit format %d/%m/%Y"),
 ("Marine", "Status (MR-1002, 1005, 1008)", "Leading/trailing spaces ('DELIVERED ', ' DELIVERED', 'IN TRANSIT ')",
  "'DELIVERED ' != 'DELIVERED' so group-by splits one status into 2 groups; counts per status are wrong", "Staging", "TRIM the text"),
 ("Marine", "Status (all rows)", "UPPERCASE while the contract wants 'Delivered / In Transit / Delayed'",
  "Case-sensitive joins/filters miss rows; reports show shouting text", "Staging", "Title-case after trimming"),
 ("Marine", "Weight_KG", "Already kg (fine) but this is the only file where it is - the other two differ",
  "Appending without a unit check hides the fact that sources use different units", "Staging", "Document the unit per source, convert in staging"),
 ("Rail", "Rows 1-2 of the sheet", "Two title rows above the header ('Pakistan Railways Freight...', 'Period...')",
  "Header is read from the wrong row; columns become Unnamed or the title becomes the header", "Staging", "Remove top 2 rows, then promote header"),
 ("Rail", "Weight (Tons)", "Weight is in TONS not kg (32.5 = 32,500 kg)",
  "Finance's mistake: rail adds only 295.25 kg-equivalents, total drops to 283.70 t", "Staging", "Multiply by 1000 into WeightKG"),
 ("Rail", "Consignment No", "ID is a plain number (45012), no prefix",
  "Can clash with ids from other modes, can't tell which system it came from, number type loses leading zeros", "Staging", "Build 'RL-' & ID as text"),
 ("Rail", "Status (45013, 45015, 45017, 45019 ...)", "Mixed case ('delivered', 'DELAYED', 'in Transit') and spaces (' Delivered', 'Delivered ')",
  "Same status shows up under many spellings", "Staging", "TRIM + title-case"),
 ("Rail", "Column names", "Names like 'Consignment No', 'Freight Charges (PKR)' contain spaces/brackets/units",
  "Append by name creates extra columns if the rename is not done identically in each query", "Staging", "Rename to the contract names"),
 ("Road", "trip_id", "IDs are lowercase (rd-501)", "Doesn't match the contract prefix RD-; case-sensitive lookups fail", "Staging", "UPPERCASE"),
 ("Road", "Row 6 / 7 (rd-505 twice)", "Exact duplicate trip",
  "Weight 6,100 kg and cost 47,000 counted twice -> Sales' total of 584.75 t", "Staging", "Drop duplicate, keep first, log it"),
 ("Road", "Row 15", "Completely blank row (,,,,,,)", "Loads as a null shipment; breaks 'not blank' rule on ShipmentID, inflates the row count", "Staging", "Remove blank row"),
 ("Road", "Row 16", "'TOTAL' row with weight 83,000 and cost 652,000 - not a shipment",
  "Double counts all of road -> Operations' total of 661.65 t", "Staging", "Remove, but keep it in the audit log"),
 ("Road", "trip_date", "Dates stored as text (yyyy-mm-dd)", "Sorts alphabetically, date functions don't work", "Staging", "Convert to real date"),
 ("Road", "status (rd-503, 509, 511)", "Lowercase + spaces ('in transit ', ' in transit')", "Status groups split", "Staging", "TRIM + title-case"),
 ("All", "Column names", "Same fact has 3 different names (Weight_KG / Weight (Tons) / weight_kg)", "Union by name gives 3x the columns, half empty", "Staging", "Map to one contract schema"),
 ("All", "Date formats", "3 different date formats (dd/mm/yyyy text, real Excel date, yyyy-mm-dd text)", "Mixed types in the ShipDate column of the warehouse", "Staging", "One DATE type"),
 ("All", "No source tag", "Nothing in the rows says which system the record came from", "Once appended you can't trace a row back to its file (no lineage)", "Staging", "Add SourceSystem audit column"),
 ("All", "Cities / stations", "Different naming ('Karachi Port', 'Karachi Cantt', 'Karachi'; 'Peshawar' vs 'Peshawar Cantt')",
  "Same city counted as different locations in route/geography reports", "Storage", "Conformed location dimension with a mapping table"),
]
t1 = pd.DataFrame(issues, columns=["Source file", "Where", "Problem found", "What goes wrong if loaded as-is?", "Fix in tier", "How to fix"])
t1.index = np.arange(1, len(t1) + 1)
t1.index.name = "#"
print("Issues found:", len(t1), "(task asks for at least 10)")
t1

## Task 2 — The Stovepipe Game

Three departments, each with their own mini-warehouse and their own rules:

| Dept | Rule |
|---|---|
| Operations | removed the duplicate rd-505, **forgot the TOTAL row**, rail converted to kg correctly |
| Finance | cleaned road correctly, **loaded rail tons as if they were kg** |
| Sales | removed the TOTAL row, **kept the duplicate trip**, rail converted correctly |

I need the raw sums per source first.

In [ ]:
marine_kg = marine_raw["Weight_KG"].sum()
rail_tons = rail_raw["Weight (Tons)"].sum()
rail_kg   = rail_tons * 1000

road_w = pd.to_numeric(road_raw["weight_kg"], errors="coerce")   # blanks become NaN
road_all_rows_kg = road_w.sum()                                    # includes duplicate + TOTAL row
road_total_row   = road_w[road_raw["trip_id"] == "TOTAL"].sum()
road_dup_kg      = road_w[dup_mask].sum()
road_clean_kg    = road_all_rows_kg - road_total_row - road_dup_kg

print(f"Marine kg                : {marine_kg:>12,.0f}")
print(f"Rail tons                : {rail_tons:>12,.2f}")
print(f"Rail kg (x1000)          : {rail_kg:>12,.0f}")
print(f"Road, every raw line     : {road_all_rows_kg:>12,.0f}")
print(f"   - TOTAL row           : {road_total_row:>12,.0f}")
print(f"   - duplicate rd-505    : {road_dup_kg:>12,.0f}")
print(f"Road, clean              : {road_clean_kg:>12,.0f}")

In [ ]:
# each department's number, built only from its own rule
ops_kg     = marine_kg + rail_kg + (road_all_rows_kg - road_dup_kg)        # keeps TOTAL, drops dup
finance_kg = marine_kg + rail_tons + road_clean_kg                        # rail NOT x1000
sales_kg   = marine_kg + rail_kg + (road_all_rows_kg - road_total_row)    # drops TOTAL, keeps dup
truth_kg   = marine_kg + rail_kg + road_clean_kg                          # one staging area

t2 = pd.DataFrame({
    "Department": ["Operations", "Finance", "Sales", "Official truth"],
    "Flawed rule": ["kept TOTAL row", "rail tons read as kg", "kept duplicate rd-505", "one staging area, one set of rules"],
    "Total kg": [ops_kg, finance_kg, sales_kg, truth_kg],
})
t2["Tonnes"] = t2["Total kg"] / 1000
t2["Diff from truth (t)"] = t2["Tonnes"] - truth_kg / 1000
t2.round(2)

In [ ]:
# the README gives the expected values, so assert them
assert round(ops_kg/1000, 2)     == 661.65
assert round(finance_kg/1000, 2) == 283.70
assert round(sales_kg/1000, 2)   == 584.75
assert truth_kg == 578650
print("all four totals match the README")

## Task 3 — Staging: clean the three files into the data contract

Target schema from the README / Start Here tab:

| Column | Type | Rule |
|---|---|---|
| ShipmentID | text | not blank, unique, prefix MR- / RL- / RD- |
| ShipDate | date | real date |
| Origin, Destination | text | |
| WeightKG | number | > 0, kilograms only |
| CostPKR | number | > 0 |
| Status | text | exactly Delivered / In Transit / Delayed |
| SourceSystem | text | Marine / Rail / Road (audit) |

I'm writing one function per source so the transformations are visible and repeatable.

In [ ]:
CONTRACT_COLS = ["ShipmentID", "ShipDate", "Origin", "Destination",
                 "WeightKG", "CostPKR", "Status", "SourceSystem"]
VALID_STATUS = ["Delivered", "In Transit", "Delayed"]

def clean_status(s):
    # trim first, then fix the case. 'in Transit' -> 'In Transit'
    return s.str.strip().str.title()

### 3a — Marine

In [ ]:
def stage_marine(raw):
    df = raw.copy()
    out = pd.DataFrame({
        "ShipmentID":   df["Shipment_ID"].str.strip().str.upper(),
        # explicit dd/mm/yyyy format, this is the UK-locale fix
        "ShipDate":     pd.to_datetime(df["Ship_Date"], format="%d/%m/%Y"),
        "Origin":       df["Origin_Port"].str.strip(),
        "Destination":  df["Dest_Port"].str.strip(),
        "WeightKG":     df["Weight_KG"].astype(float),     # already kg, no conversion
        "CostPKR":      df["Cost_PKR"].astype(float),
        "Status":       clean_status(df["Status"]),
        "SourceSystem": "Marine",
    })
    return out

marine_stg = stage_marine(marine_raw)
marine_stg

In [ ]:
# trap check from the workbook: MR-1003 must be 03-Feb, MR-1012 must be 13-Feb
chk = marine_stg.set_index("ShipmentID").loc[["MR-1003", "MR-1012"], "ShipDate"]
print(chk.dt.strftime("%d-%b-%Y"))
assert chk["MR-1003"] == pd.Timestamp(2026, 2, 3)
assert chk["MR-1012"] == pd.Timestamp(2026, 2, 13)

### 3b — Rail

In [ ]:
def stage_rail(raw):
    df = raw.copy()
    out = pd.DataFrame({
        # numeric id -> text with the RL- prefix
        "ShipmentID":   "RL-" + df["Consignment No"].astype(int).astype(str),
        "ShipDate":     pd.to_datetime(df["Dispatch Date"]),
        "Origin":       df["From Station"].str.strip(),
        "Destination":  df["To Station"].str.strip(),
        # tons -> kg. round() to kill float noise like 28.75*1000 = 28750.000000000004
        "WeightKG":     (df["Weight (Tons)"] * 1000).round(0),
        "CostPKR":      df["Freight Charges (PKR)"].astype(float),
        "Status":       clean_status(df["Status"]),
        "SourceSystem": "Rail",
    })
    return out

rail_stg = stage_rail(rail_raw)
rail_stg

In [ ]:
# check from the workbook hint: RL-45012 should weigh 32,500 kg
print(rail_stg.loc[rail_stg["ShipmentID"] == "RL-45012", "WeightKG"].iloc[0])
assert rail_stg.loc[rail_stg["ShipmentID"] == "RL-45012", "WeightKG"].iloc[0] == 32500

### 3c — Road

Road is different from the other two. The workbook says: don't delete the junk rows, **mark them and explain**, so the audit trail survives. So I transform every line first, then add `Keep?` and `Reason if N` columns.

In [ ]:
def stage_road(raw):
    df = raw.copy()
    df = df.apply(lambda c: c.str.strip())               # blanks become '' after the strip

    out = pd.DataFrame({
        "ShipmentID":   df["trip_id"].str.upper(),
        "ShipDate":     pd.to_datetime(df["trip_date"], format="%Y-%m-%d", errors="coerce"),
        "Origin":       df["from_city"].replace("", np.nan),
        "Destination":  df["to_city"].replace("", np.nan),
        "WeightKG":     pd.to_numeric(df["weight_kg"], errors="coerce"),
        "CostPKR":      pd.to_numeric(df["cost_pkr"], errors="coerce"),
        "Status":       clean_status(df["status"]).replace("", np.nan),
        "SourceSystem": "Road",
    })

    # --- decide keep / remove, in this order: blank -> total -> duplicate
    is_blank = (df == "").all(axis=1)
    is_total = ~out["ShipmentID"].str.match(r"^RD-\d+$") & ~is_blank
    # duplicate = same full record as an earlier row. check on the cleaned columns
    is_dup   = out.duplicated(subset=CONTRACT_COLS, keep="first") & ~is_blank & ~is_total

    out["Keep?"] = "Y"
    out["Reason if N"] = ""
    out.loc[is_blank, ["Keep?", "Reason if N"]] = ["N", "Blank row - no data at all"]
    out.loc[is_total, ["Keep?", "Reason if N"]] = ["N", "TOTAL row - a subtotal from the source, not a shipment"]
    out.loc[is_dup,   ["Keep?", "Reason if N"]] = ["N", "Duplicate of an earlier trip (same id, date, route, weight, cost)"]
    return out

road_stg_all = stage_road(road_raw)
road_stg_all

In [ ]:
# the 3 removed rows, one of each kind
road_stg_all[road_stg_all["Keep?"] == "N"][["ShipmentID", "WeightKG", "CostPKR", "Keep?", "Reason if N"]]

In [ ]:
# the clean road table = only the Y rows, contract columns only
road_stg = road_stg_all.loc[road_stg_all["Keep?"] == "Y", CONTRACT_COLS].reset_index(drop=True)
road_stg

### Data contract validation

Rules from the Start Here tab, tested as code instead of eyeballing. If any rule fails the notebook stops with an error.

In [ ]:
def validate(df, name):
    errs = []
    if df["ShipmentID"].isna().any() or (df["ShipmentID"].str.strip() == "").any(): errs.append("blank ShipmentID")
    if df["ShipmentID"].duplicated().any():                                        errs.append("duplicate ShipmentID")
    if not df["ShipmentID"].str.match(r"^(MR|RL|RD)-\d+$").all():                  errs.append("bad ID prefix/format")
    if df["ShipDate"].isna().any():                                                errs.append("missing/invalid date")
    if not (df["WeightKG"] > 0).all():                                             errs.append("weight <= 0 or missing")
    if not (df["CostPKR"] > 0).all():                                              errs.append("cost <= 0 or missing")
    if not df["Status"].isin(VALID_STATUS).all():                                  errs.append("status outside the 3 allowed values")
    if df["SourceSystem"].isna().any():                                            errs.append("missing SourceSystem")
    print(f"{name:7s}", "OK" if not errs else "FAILED -> " + ", ".join(errs))
    return not errs

ok = [validate(marine_stg, "Marine"), validate(rail_stg, "Rail"), validate(road_stg, "Road")]
assert all(ok), "contract broken, fix staging before going on"

In [ ]:
# and to show the validation actually catches things: run it on the UN-cleaned road rows
try_raw = pd.DataFrame({
    "ShipmentID": road_raw["trip_id"], "ShipDate": pd.to_datetime(road_raw["trip_date"], errors="coerce"),
    "WeightKG": pd.to_numeric(road_raw["weight_kg"], errors="coerce"),
    "CostPKR": pd.to_numeric(road_raw["cost_pkr"], errors="coerce"),
    "Status": road_raw["status"], "SourceSystem": "Road",
})
validate(try_raw, "Raw Rd")

## Task 4 — Reconcile: prove 34 rows and 578,650 kg

Rule from the workbook: **Raw lines − Removed = Loaded**. Every removed row must have a reason.

In [ ]:
def n_raw(src):
    return {"Marine": len(marine_raw), "Rail": len(rail_raw), "Road": len(road_raw)}[src]

stg = {"Marine": marine_stg, "Rail": rail_stg, "Road": road_stg}
expected_rows = {"Marine": 12, "Rail": 10, "Road": 12}
expected_kg   = {"Marine": 206500, "Rail": 295250, "Road": 76900}

rows = []
for src in ["Marine", "Rail", "Road"]:
    raw_n    = n_raw(src)
    loaded   = len(stg[src])
    removed  = raw_n - loaded
    kg       = stg[src]["WeightKG"].sum()
    rows.append([src, raw_n, removed, loaded, expected_rows[src], kg, expected_kg[src],
                 "OK" if (loaded == expected_rows[src] and abs(kg - expected_kg[src]) < 0.5) else "FIX IT"])

recon = pd.DataFrame(rows, columns=["Source", "Raw lines", "Removed", "Loaded", "Expected rows", "Weight (kg)", "Expected kg", "Check"])
tot = recon[["Raw lines", "Removed", "Loaded", "Expected rows", "Weight (kg)", "Expected kg"]].sum()
tot_check = "OK" if (tot["Loaded"] == 34 and abs(tot["Weight (kg)"] - 578650) < 0.5 and tot["Raw lines"] - tot["Removed"] == tot["Loaded"]) else "FIX IT"
recon.loc[len(recon)] = ["Total", *tot.values, tot_check]
recon

In [ ]:
print("Total freight (tonnes):", recon.loc[recon["Source"] == "Total", "Weight (kg)"].iloc[0] / 1000)
assert (recon["Check"] == "OK").all()

In [ ]:
# status check (expected 22 / 8 / 4, nothing else)
all_stg = pd.concat([marine_stg, rail_stg, road_stg], ignore_index=True)

status_tbl = pd.crosstab(all_stg["Status"], all_stg["SourceSystem"])
status_tbl["Total"] = status_tbl.sum(axis=1)
status_tbl = status_tbl.reindex(VALID_STATUS)
status_tbl["Expected"] = [22, 8, 4]
status_tbl["Check"] = np.where(status_tbl["Total"] == status_tbl["Expected"], "OK", "FIX IT")
print("Other status values:", (~all_stg["Status"].isin(VALID_STATUS)).sum())
status_tbl

In [ ]:
# weight by mode in tonnes (README: Marine 206.5 / Rail 295.25 / Road 76.9)
by_mode = all_stg.groupby("SourceSystem")["WeightKG"].sum().div(1000).rename("Tonnes").to_frame()
by_mode.loc["Total"] = by_mode["Tonnes"].sum()
by_mode

In [ ]:
assert status_tbl["Check"].eq("OK").all()
assert by_mode.loc["Total", "Tonnes"] == 578.65
print("everything reconciles: 37 raw - 3 removed = 34 loaded, 578,650 kg")

In [ ]:
# audit log of what got removed - this is the lineage evidence for the 3 road rows
audit_removed = road_stg_all.loc[road_stg_all["Keep?"] == "N", ["ShipmentID", "WeightKG", "CostPKR", "Reason if N"]].copy()
audit_removed.insert(0, "Source", "Road")
audit_removed.insert(1, "Raw line #", audit_removed.index)
audit_removed

## Task 5 — Trace the lineage of one number

Metric: **Total Freight = 578.65 tonnes** (the card on the CEO dashboard). I'm following it back, tier by tier.

In [ ]:
lineage = pd.DataFrame([
 ["1 · Source files",
  "3 files, 3 formats: marine CSV (kg, dd/mm/yyyy), rail XLSX with 2 title rows (tons), road CSV with duplicate/blank/TOTAL rows. 37 raw lines. Source totals: 206,500 kg + 295.25 t + road lines.",
  "Nothing. Files are only read, never edited.",
  "File received check: file name, row count (12 / 10 / 15), file date.",
  "File name, load timestamp, original row numbers."],
 ["2 · Staging",
  "Raw copy of each file, then a transformed copy in the contract schema (8 columns).",
  "Skip 2 title rows (rail), dd/mm/yyyy -> date, trim + title-case status, RL-/RD-/MR- ids, **rail tons x 1000 -> kg**, flag blank / TOTAL / duplicate road rows as Keep=N with a reason.",
  "Data contract rules (id unique & prefixed, date valid, weight > 0, status in 3 values). Raw - Removed = Loaded. 'Other status values' = 0.",
  "SourceSystem column, Keep?/Reason table (3 removed road rows), row counts per source (37 -> 34)."],
 ["3 · Storage (integrated)",
  "One table, 34 rows, same schema for all modes (Fact_Shipments or an EDW shipment table).",
  "Append of the 3 staged tables by column name; surrogate keys for date / route / mode; cities mapped to one location list.",
  "Row count = 34, SUM(WeightKG) = 578,650, no duplicate ShipmentID, no nulls in keys.",
  "Load batch id, load date, source system, rows in / rows rejected."],
 ["4 · Presentation",
  "Star schema: Fact_Shipments + Dim_Date, Dim_Mode, Dim_Status, Dim_Location. Measure Total Weight (t) = SUM(WeightKG)/1000.",
  "Model relationships + one DAX/SQL measure that divides by 1000 (the only unit step here).",
  "Measure total on the model = staging total = 578.65.",
  "Measure definition, who owns it (data steward), model refresh time."],
 ["5 · Report",
  "CEO dashboard card: 578.65 t. Plus charts by mode (206.5 / 295.25 / 76.9) and by status (22 / 8 / 4).",
  "Formatting only (number format, label 'tonnes').",
  "Card = table total; spot check one slice (e.g. Rail = 295.25).",
  "Report name, filter state, last refresh, link back to the data dictionary."],
], columns=["Tier", "What the data looks like here", "What changes here (transformation)", "Quality check you'd place here", "Audit / lineage info you'd keep"])
lineage.set_index("Tier")

## Task 6 — Kimball bus matrix

Rows = business processes (one star schema each), columns = dimensions. X = the process needs that dimension. Row 1 (Shipments) is the example given in the workbook.

In [ ]:
dims = ["Date", "Customer", "Route", "Vehicle", "Driver", "Station / Port", "Commodity"]

matrix = {
 "Shipments":             ["Date", "Customer", "Route", "Vehicle", "Station / Port", "Commodity"],   # given
 "Billing & Invoicing":   ["Date", "Customer", "Route", "Commodity"],
 "Fleet Maintenance":     ["Date", "Vehicle", "Station / Port"],
 "Fuel Purchases":        ["Date", "Vehicle", "Driver", "Station / Port"],
 "Driver Trips & Hours":  ["Date", "Route", "Vehicle", "Driver"],
 "Customer Complaints":   ["Date", "Customer", "Route", "Driver", "Commodity"],
}

bus = pd.DataFrame("", index=matrix.keys(), columns=dims)
for proc, used in matrix.items():
    bus.loc[proc, used] = "X"

bus["Dims used"] = (bus[dims] == "X").sum(axis=1)
bus.loc["Processes using this dim"] = [*(bus[d].eq("X").sum() for d in dims), ""]
bus

## Task 7 — Inmon, Kimball or Hybrid? (10 scenarios)

In [ ]:
t7 = pd.DataFrame([
 (1,  "A bank must integrate 40 systems into one enterprise model for regulators before anyone builds reports.",
      "Inmon",  "Enterprise-wide integrated model first (top-down), accuracy and consistency before reporting."),
 (2,  "PakFreight's COO wants a useful shipments dashboard within 3 months.",
      "Kimball", "One star schema for the Shipments process delivers value fast (bottom-up)."),
 (3,  "The core warehouse is stored in normalised third normal form (3NF).",
      "Inmon",  "Inmon's EDW is normalised 3NF; Kimball's presentation area is dimensional."),
 (4,  "Data marts ARE the presentation area, tied together by conformed dimensions.",
      "Kimball", "In Kimball the marts are the warehouse, glued by the bus architecture."),
 (5,  "The bus matrix is planned for the whole company, but built one process at a time.",
      "Kimball", "That's literally the bus matrix approach: plan enterprise-wide, deliver incrementally."),
 (6,  "Departments get dependent marts that are derived from the central EDW.",
      "Inmon",  "Dependent marts fed from the EDW are the Corporate Information Factory design."),
 (7,  "Biggest risk: a long wait before users see any value.",
      "Inmon",  "Big enterprise model upfront means a long time before the first report."),
 (8,  "Biggest risk: sliding back into stovepipes if teams ignore conformance.",
      "Kimball", "Without conformed dimensions the separate marts drift apart again."),
 (9,  "Start from the data; users discover real requirements after they see it (CLDS).",
      "Inmon",  "Data-driven development, CLDS is the reverse of SDLC and is associated with Inmon's approach."),
 (10, "Bronze raw files -> Silver cleaned & integrated tables -> Gold star schemas in Microsoft Fabric.",
      "Hybrid", "Silver = integrated layer (Inmon-ish), Gold = dimensional stars (Kimball-ish)."),
], columns=["#", "Statement / scenario", "Inmon · Kimball · Hybrid", "Why? (one line)"]).set_index("#")
t7

In [ ]:
t7["Inmon · Kimball · Hybrid"].value_counts()

## Task 8 — Draw both architectures (Inmon CIF vs Kimball bus)

The workbook says this one is done on paper. I'm drawing it with matplotlib so it's in the notebook too (the hand drawing can copy this layout).

In [ ]:
def box(ax, x, y, w, h, text, color="#e8eef7", fs=9, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc=color, ec="#333", lw=1.2))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal")

def arrow(ax, x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", lw=1.4, color="#333"))

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

# ---------- Inmon CIF
ax = axes[0]; ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("Inmon — Corporate Information Factory (top-down)", fontweight="bold")
srcs = ["Marine\nCSV", "Rail\nXLSX", "Road\nCSV"]
for i, s in enumerate(srcs):
    box(ax, 0.3 + i*3.2, 0.3, 2.6, 1.3, s, "#f6e7c8")
box(ax, 0.3, 2.3, 9.3, 1.3, "Staging / ETL  (clean, convert tons->kg, dedupe)", "#e3e3e3")
box(ax, 0.3, 4.3, 9.3, 1.5, "Enterprise Data Warehouse (EDW)\nnormalised 3NF · subject-oriented · integrated · time-variant · non-volatile", "#cfe0f5", bold=True)
for i, s in enumerate(["Ops mart\n(dependent)", "Finance mart\n(dependent)", "Sales mart\n(dependent)"]):
    box(ax, 0.3 + i*3.2, 6.6, 2.6, 1.3, s, "#d6efd6")
box(ax, 0.3, 8.5, 9.3, 1.0, "BI tools / reports / dashboards", "#f3d1d1")
for i in range(3):
    arrow(ax, 1.6 + i*3.2, 1.6, 1.6 + i*3.2, 2.3)
    arrow(ax, 1.6 + i*3.2, 5.8, 1.6 + i*3.2, 6.6)
    arrow(ax, 1.6 + i*3.2, 7.9, 1.6 + i*3.2, 8.5)
arrow(ax, 4.9, 3.6, 4.9, 4.3)

# ---------- Kimball bus
ax = axes[1]; ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("Kimball — Dimensional bus (bottom-up)", fontweight="bold")
for i, s in enumerate(srcs):
    box(ax, 0.3 + i*3.2, 0.3, 2.6, 1.3, s, "#f6e7c8")
box(ax, 0.3, 2.3, 9.3, 1.3, "Staging area (not for users: clean, conform, no queries)", "#e3e3e3")
box(ax, 0.3, 4.3, 9.3, 3.6, "", "#eef6ff")
ax.text(4.95, 7.55, "Presentation area — dimensional data marts (stars)", ha="center", fontsize=9, fontweight="bold")
for i, s in enumerate(["Shipments\nstar", "Billing\nstar", "Fleet\nstar"]):
    box(ax, 0.6 + i*3.1, 5.9, 2.5, 1.2, s, "#cfe0f5")
box(ax, 0.6, 4.5, 9.0 - 0.6 - 0.0, 0.9, "Conformed dimensions (bus):  Date · Customer · Route · Vehicle · Driver · Station", "#fff2b3", bold=True, fs=8.5)
box(ax, 0.3, 8.5, 9.3, 1.0, "BI tools / reports / dashboards", "#f3d1d1")
for i in range(3):
    arrow(ax, 1.6 + i*3.2, 1.6, 1.6 + i*3.2, 2.3)
    arrow(ax, 1.6 + i*3.2, 7.9, 1.6 + i*3.2, 8.5)
arrow(ax, 4.9, 3.6, 4.9, 4.3)

plt.tight_layout()
plt.show()

| | Inmon (CIF) | Kimball (bus) |
|---|---|---|
| Approach | top-down, enterprise first | bottom-up, process by process |
| Central store | normalised EDW (3NF) | no separate EDW: the marts *are* the warehouse |
| Marts | **dependent**, fed from the EDW | stars tied together by **conformed dimensions** |
| Time to first value | slow | fast |
| Main risk | long wait / big upfront cost | stovepipes if conformance is ignored |

# Part B  Theory

Short answers, kept tied to the PakFreight case where it makes sense.

## B1. The three tiers of a data warehouse architecture

| Tier | Also called | What it does | PakFreight example |
|---|---|---|---|
| Bottom | Data source & staging / back-end | Sources + ETL: extract, clean, transform, load | 3 shipment files, staging with the contract schema |
| Middle | Storage / warehouse server | Integrated, historical, subject-oriented data (EDW, or dimensional marts) | Fact_Shipments with 34 rows |
| Top | Presentation / front-end | BI tools, dashboards, OLAP, ad-hoc queries | CEO dashboard card with 578.65 t |

Lifecycle thinking: the tiers are the structure, the lifecycle (below) is the order of work used to build it.

## B2. Inmon's four properties of a data warehouse

1. Subject-oriented: organised around business subjects (shipments, customers) rather than around applications (marine system, rail system).
2. Integrated: data from different sources made consistent: one naming convention, one unit (kg), one date format, one status list.
3. Time-variant: keeps history, every record is tied to a time, so you can compare Feb vs Mar.
4. Non-volatile: data is loaded and read, not updated in place. New facts are added, old ones aren't overwritten.

In this case the 3 files being appended into one kg/date/status-clean table demonstrates *integrated*; the `ShipDate` column plus history is *time-variant*; and keeping removed rows in the audit log instead of deleting is the non-volatile mindset.

## B3. EDW vs ODS

| | EDW | ODS (Operational Data Store) |
|---|---|---|
| Purpose | strategic / historical analysis | operational reporting, current state |
| Data | integrated history, years | current or very recent (days) |
| Updates | batch loads, non-volatile | frequent / near real-time, data can be overwritten |
| Granularity | detailed + summarised | mostly detailed |
| Users | analysts, management | operational staff, call-centre, dispatchers |
| Example | "tonnage trend by mode, last 3 years" | "which consignments are in transit right now?" |

## B4. Independent vs dependent data marts

- **Independent mart**: built straight from the source systems for one department, no shared warehouse. Quick and cheap, but every team applies its own rules. This is exactly the stovepipe problem in Task 2 (Ops 661.65 / Finance 283.70 / Sales 584.75).
- **Dependent mart**: a subset of the EDW (or of conformed dimensions) built for a department. All rules were applied once upstream, so all marts agree. Costs more upfront and needs the central layer first.

## B5. Inmon vs Kimball

| | Inmon | Kimball |
|---|---|---|
| Direction | top-down | bottom-up |
| Model | normalised 3NF EDW, then marts | dimensional stars with conformed dims |
| Build | big integrated warehouse first | one business process at a time |
| Data-ownership | IT-led enterprise model | business-process-led |
| Strength | single version of truth, consistency | speed, usability, query performance |
| Weakness | slow, expensive, late value | needs discipline on conformance |
| Typical today | hybrid: integrated layer (silver) + stars (gold) | |

## B6. The DW lifecycle phases

Project planning -> business requirements -> (in parallel: technology track, data track, BI application track) -> deployment -> maintenance and growth.
- Technology track: architecture, product selection, installation
- Data track: dimensional modelling, physical design, ETL design and development
- BI application track: specification and development of reports / dashboards
For PakFreight the first pass would be: plan (3 totals problem), requirements (COO wants volume + cost + status), data track (staging + Fact_Shipments), BI track (CEO dashboard), then deploy, then grow with Billing and Fleet.

## B7. SDLC vs CLDS

| | SDLC | CLDS (reverse life cycle) |
|---|---|---|
| Starts with | requirements | the data |
| Flow | requirements -> design -> build -> test -> deploy | data -> program -> analyse results -> understand requirements -> repeat |
| Fits | operational systems with known needs | DW / analytics where users only know what they need after they've seen the data |
| Iteration | mostly linear | cyclical, discovery driven |

In the PakFreight activity we followed CLDS: look at the files first, find the problems, then work out what the contract should be.

## B8. Data latency

How long after an event happens it becomes visible in a report. Batch (nightly), micro-batch (hourly), near real-time, real-time. Higher freshness costs more (streaming ETL, more complex checks) so latency should be matched to the decision: monthly tonnage reports don't need real-time, a dispatcher tracking delayed consignments does. With 3 files delivered monthly, PakFreight is on batch/monthly latency right now.

## B9. Schema drift

When a source quietly changes its structure: a column renamed, added, removed, or its type/unit changes (e.g. rail starts sending kg, not tons, or `Weight (Tons)` becomes `Weight_Tons`). Appending by *name* in Power BI would silently create an extra column and leave the old one null. Defences: explicit column mapping in staging, schema checks at load (expected columns, types), alerts when a file does not match, and unit checks (a rail average of 29 "kg" is obviously wrong).

## B10. Data lineage

The documented path of a value from source to report: where it came from, every transformation, and who owns each step. Task 5 showed it for 578.65 t. It's used for trust (explaining why numbers differ), impact analysis (what breaks if rail changes its file) and audit.

## B11. BI roles

| Role | What they do | In this case |
|---|---|---|
| Business sponsor / data owner | decides business definitions | COO decides what "one shipment" is |
| Data steward | maintains definitions & quality rules | owns status list and ID rules |
| Data architect | designs layers & models | designs the contract + star schema |
| ETL / data engineer | builds the staging & loads | writes the transformations in this notebook |
| BI developer | builds models, measures, reports | Total Weight measure, dashboard |
| Data analyst | explores & interprets | compares modes, status trends |
| DBA | performance, security, backup | looks after storage tier |

## B12. Responsibility map (who fixes what, in which tier)

| Problem | Tier where it's fixed | Who |
|---|---|---|
| Wrong date format / text dates | Staging | ETL developer |
| Tons vs kg | Staging | ETL developer, rule agreed by data steward |
| Duplicate / TOTAL / blank rows | Staging | ETL developer (rule from data steward) |
| Definition of "shipment" | Business / governance | business sponsor + data steward |
| Same city, different names | Storage (conformed dimension) | data architect |
| Total tonnage measure | Presentation | BI developer |
| Number format on the card | Report | BI developer / analyst |

# Part C — Hands-on: staging → storage → presentation

The assignment's hands-on task is done in Power BI. Same pipeline in code: staging is already built above, so next is storage (one integrated table), then presentation (star schema + measures).

## C1. Storage tier: append into one table

In [ ]:
# append by NAME. All three have exactly the contract columns, so no stray columns appear
storage = pd.concat([marine_stg, rail_stg, road_stg], ignore_index=True)
storage["WeightKG"] = storage["WeightKG"].astype(float)
storage["CostPKR"]  = storage["CostPKR"].astype(float)
print(storage.shape, "->", list(storage.columns))
storage.sort_values("ShipDate").reset_index(drop=True)

In [ ]:
# the contract again, now on the integrated table
assert validate(storage, "Storage")
assert len(storage) == 34 and storage["WeightKG"].sum() == 578650
assert storage["ShipmentID"].is_unique

In [ ]:
# date shown like the contract example (03-Feb-2026) is a display format only, the column stays a real date
storage.assign(ShipDate=storage["ShipDate"].dt.strftime("%d-%b-%Y")).head()

## C2. Presentation tier: star schema

In [ ]:
# --- dimensions with surrogate keys
dim_date = pd.DataFrame({"Date": pd.date_range(storage["ShipDate"].min(), storage["ShipDate"].max(), freq="D")})
dim_date["DateKey"]   = dim_date["Date"].dt.strftime("%Y%m%d").astype(int)
dim_date["Day"]       = dim_date["Date"].dt.day
dim_date["DayName"]   = dim_date["Date"].dt.day_name()
dim_date["Month"]     = dim_date["Date"].dt.month_name()
dim_date["Year"]      = dim_date["Date"].dt.year

dim_mode = pd.DataFrame({"Mode": ["Marine", "Rail", "Road"]})
dim_mode["ModeKey"] = dim_mode.index + 1

dim_status = pd.DataFrame({"Status": VALID_STATUS})
dim_status["StatusKey"] = dim_status.index + 1

# location dimension: origin + destination share one list (role-playing)
locs = pd.Series(pd.concat([storage["Origin"], storage["Destination"]]).unique(), name="Location").sort_values()
dim_location = locs.reset_index(drop=True).to_frame()
dim_location["LocationKey"] = dim_location.index + 1

dim_location.head(8)

In [ ]:
# --- fact table: one row per shipment, keys + measures
fact = (storage
        .merge(dim_mode,   left_on="SourceSystem", right_on="Mode")
        .merge(dim_status, on="Status")
        .merge(dim_location.rename(columns={"Location": "Origin", "LocationKey": "OriginKey"}), on="Origin")
        .merge(dim_location.rename(columns={"Location": "Destination", "LocationKey": "DestinationKey"}), on="Destination"))
fact["DateKey"] = fact["ShipDate"].dt.strftime("%Y%m%d").astype(int)

fact_shipments = fact[["ShipmentID", "DateKey", "ModeKey", "StatusKey", "OriginKey", "DestinationKey", "WeightKG", "CostPKR"]]
fact_shipments = fact_shipments.sort_values("ShipmentID").reset_index(drop=True)
print("fact rows:", len(fact_shipments), "(must stay 34, a bad join would change this)")
assert len(fact_shipments) == 34
fact_shipments.head()

In [ ]:
# --- put it in a real database (SQLite) so it behaves like a warehouse
con = sqlite3.connect(":memory:")
for name, d in {"Fact_Shipments": fact_shipments, "Dim_Date": dim_date, "Dim_Mode": dim_mode,
                "Dim_Status": dim_status, "Dim_Location": dim_location}.items():
    d2 = d.copy()
    for c in d2.columns:
        if str(d2[c].dtype).startswith("datetime"):
            d2[c] = d2[c].dt.strftime("%Y-%m-%d")
    d2.to_sql(name, con, index=False)

# measure: Total Weight (t) - same thing the DAX card would show
pd.read_sql("""
    SELECT ROUND(SUM(WeightKG) / 1000.0, 2) AS Total_Weight_Tonnes,
           COUNT(*)                         AS Shipments,
           SUM(CostPKR)                     AS Total_Cost_PKR
    FROM Fact_Shipments
""", con)

In [ ]:
# by mode, with a cost per kg since finance will ask for it
pd.read_sql("""
    SELECT m.Mode,
           COUNT(*)                                   AS Shipments,
           SUM(f.WeightKG) / 1000.0                   AS Tonnes,
           SUM(f.CostPKR)                             AS Cost_PKR,
           ROUND(SUM(f.CostPKR) * 1.0 / SUM(f.WeightKG), 2) AS PKR_per_kg
    FROM Fact_Shipments f
    JOIN Dim_Mode m ON m.ModeKey = f.ModeKey
    GROUP BY m.Mode
    ORDER BY Tonnes DESC
""", con)

In [ ]:
# by status and mode
pd.read_sql("""
    SELECT s.Status, m.Mode, COUNT(*) AS Shipments
    FROM Fact_Shipments f
    JOIN Dim_Status s ON s.StatusKey = f.StatusKey
    JOIN Dim_Mode   m ON m.ModeKey   = f.ModeKey
    GROUP BY s.Status, m.Mode
""", con).pivot(index="Status", columns="Mode", values="Shipments").fillna(0).astype(int).reindex(VALID_STATUS)

In [ ]:
# top routes by weight - uses the location dim twice (origin + destination)
pd.read_sql("""
    SELECT o.Location AS Origin, d.Location AS Destination,
           COUNT(*) AS Shipments, SUM(f.WeightKG)/1000.0 AS Tonnes
    FROM Fact_Shipments f
    JOIN Dim_Location o ON o.LocationKey = f.OriginKey
    JOIN Dim_Location d ON d.LocationKey = f.DestinationKey
    GROUP BY o.Location, d.Location
    ORDER BY Tonnes DESC
    LIMIT 8
""", con)

## C3. Report tier: the dashboard visuals

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

# card-like number
ax[0].axis("off")
ax[0].text(0.5, 0.58, f"{storage['WeightKG'].sum()/1000:,.2f} t", ha="center", fontsize=34, fontweight="bold")
ax[0].text(0.5, 0.28, "Total Freight (Feb 2026)\n34 shipments", ha="center", fontsize=12)

# tonnes by mode
bm = storage.groupby("SourceSystem")["WeightKG"].sum().div(1000)
ax[1].bar(bm.index, bm.values, color=["#4c78a8", "#f58518", "#54a24b"])
for i, v in enumerate(bm.values):
    ax[1].text(i, v + 4, f"{v:,.2f}", ha="center")
ax[1].set_title("Tonnes by mode"); ax[1].set_ylabel("tonnes")

# status count
sc = storage["Status"].value_counts().reindex(VALID_STATUS)
ax[2].bar(sc.index, sc.values, color=["#54a24b", "#f2c14e", "#e45756"])
for i, v in enumerate(sc.values):
    ax[2].text(i, v + 0.3, str(v), ha="center")
ax[2].set_title("Shipments by status")

for a in ax[1:]:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# the four totals side by side, this is the picture that sells the staging area
labels = ["Finance\n(tons as kg)", "One staging\narea (truth)", "Sales\n(kept duplicate)", "Operations\n(kept TOTAL)"]
vals = [finance_kg/1000, truth_kg/1000, sales_kg/1000, ops_kg/1000]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, vals, color=["#bbbbbb", "#2a9d8f", "#bbbbbb", "#bbbbbb"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 8, f"{v:,.2f} t", ha="center")
ax.set_ylabel("tonnes"); ax.set_title("Same question, four answers")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## C4. Export the trusted table

In [ ]:
# the final contract table, as the Power BI model would load it
final = storage.sort_values(["ShipDate", "ShipmentID"]).reset_index(drop=True)
final.to_csv("PakFreight_Shipments_Clean.csv", index=False)
audit_removed.to_csv("PakFreight_Removed_Rows_Audit.csv", index=False)
print("saved", len(final), "clean rows and", len(audit_removed), "audit rows")

## Final check and conclusion

| Source | Raw | Removed | Loaded | Weight (kg) |
|---|---|---|---|---|
| Marine | 12 | 0 | 12 | 206,500 |
| Rail | 10 | 0 | 10 | 295,250 |
| Road | 15 | 3 (blank, TOTAL, duplicate) | 12 | 76,900 |
| **Total** | **37** | **3** | **34** | **578,650** |

Status: Delivered 22 · In Transit 8 · Delayed 4.

The three "wrong" totals were never arithmetic mistakes, they came from three sets of rules. Once the cleaning (title rows, dates, trim/case, tons→kg, duplicate/blank/TOTAL rows) lives in a single staging layer, all three teams get the same 578.65 t.